<a href="https://colab.research.google.com/github/SLCFLAB/Fintech2026-1/blob/main/DL_day11/11_2_longhorizon_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11-2. Transformer 계열 모델의 장기 시계열 예측 비교

이 실습은 `NeuralForecast` 라이브러리로 **Informer, Autoformer, PatchTST**를 학습하고, ETTm2 데이터에서 예측을 비교합니다. `11_1`이 작은 Encoder를 직접 읽는 실습이라면, 이번에는 **데이터 형식·모델 설정·시간순 평가·결과 해석**에 집중합니다.

**편집 범위:** 기존 코드 셀은 변경하지 않았습니다. 영어 설명을 한국어로 정리하고, 각 코드의 역할과 구현·평가 주의사항을 추가했습니다. 전체 데이터 다운로드와 세 모델의 학습은 이번 편집 과정에서 실행하지 않았으므로 새로운 성능 결과를 제시하는 파일은 아닙니다.

### 실습 목표와 범위

1. ETTm2를 라이브러리가 요구하는 long format으로 이해합니다.
2. `input_size`와 예측 길이 `h`를 구분합니다.
3. 세 모델을 같은 데이터 구간에서 학습하고 여러 예측 기준시점에서 평가합니다.
4. OT 예측 그래프와 전체 데이터 MAE가 평가하는 범위를 구분합니다.

Transformer는 Self-Attention을 이용해 입력 구간 내의 관계를 표현할 수 있습니다. 긴 입력에서는 Attention 계산 비용이 커지며, 여러 미래 시점을 예측하는 과제에는 다양한 시계열 전용 구조가 제안되었습니다. 그렇다고 Transformer가 모든 데이터에서 다른 모델보다 정확한 것은 아닙니다.

이 코드는 각 변수의 과거값으로 그 변수의 미래를 예측하는 설정입니다. 여러 `unique_id`를 함께 학습하는 **전역 모델(global model)**과, 변수들을 한 입력에 동시에 넣어 변수 간 관계를 학습하는 **다변량 입력 모델**은 구분해야 합니다. 여기서는 외생변수를 별도로 전달하지 않습니다.

원본 튜토리얼의 “단변량이 항상 더 빠르고 정확하다”, “논문 결과를 능가한다”는 취지의 문구는 **특정 실험의 보고**로만 읽어야 합니다. 아래 코드를 실행하지 않고 그 우열을 확인했다고 볼 수 없습니다. [Nixtla 원본 튜토리얼](https://nixtlaverse.nixtla.io/neuralforecast/docs/tutorials/longhorizon_transformers.html)

### 강의의 Attention과 시계열 예측 모델 연결하기

기준 자료는 **D06 Transformer2026R.pdf**이며 아래 쪽수는 PDF 뷰어 기준입니다.

| 강의 개념 | PDF 쪽 | 이번 실습에서의 연결 |
|---|---|---|
| Q/K/V와 가중합 | 51–57 | 시점 또는 패치 표현에서 관련 정보를 결합 |
| Multi-Head Attention | 60–68 | 서로 다른 투영 공간에서 관계를 학습하는 기본 발상 |
| FFN, 잔차 연결, 정규화 | 74–75, 89 | 각 모델 내부 블록을 이해하는 배경 |
| 위치 정보 | 76–81 | 수치값뿐 아니라 시간상의 순서도 표현해야 함 |
| 긴 시퀀스의 Attention 비용 | 101–103 | 희소화·패치화 등 효율적 구조의 동기 |

기본 Attention은 $\operatorname{softmax}(QK^\top/\sqrt{d_k})V$로 정보를 결합합니다. 하지만 이번 모델들이 모두 이 기본 연산만 그대로 반복하는 것은 아닙니다. Autoformer는 Auto-Correlation 등 시계열에 맞춘 연산을 사용합니다. RoPE·GQA·FlashAttention은 강의자료에 등장하더라도 **이 노트북에서 직접 설정·비교하는 대상은 아닙니다.**

### 실행 환경과 진행 순서

설치 → 데이터 로드 → 모델 생성 → `cross_validation`으로 학습·예측 → 그래프 → MAE 순서로 위에서부터 실행하세요. 모델 생성 셀까지만 실행하면 학습은 아직 시작되지 않습니다.

처음 실행할 때 패키지 설치와 데이터 다운로드를 위한 인터넷 연결이 필요합니다. 세 모델의 학습 및 많은 예측 윈도우 계산으로 시간이 걸릴 수 있어 GPU 환경이 유용하지만, 실제 사용 장치는 학습 로그에서 확인해야 합니다. 짧은 수업이라면 모델 하나와 작은 학습 step으로 동작을 먼저 확인하는 식으로 별도 실행 설정을 준비할 수 있습니다. **현재 코드는 세 모델·최대 1000 step 설정을 유지합니다.**

첫머리의 Colab 배지는 GitHub 원본을 엽니다. **이 한국어 설명 수정본을 사용하려면 내려받은 파일을 Colab에 직접 업로드하세요.**

원본의 추가 실행 링크: [Nixtla Colab 예제](https://colab.research.google.com/github/Nixtla/neuralforecast/blob/main/nbs/examples/LongHorizon_with_Transformers.ipynb)

## 1. 라이브러리 설치

첫 번째 셀은 PyTorch 관련 패키지 버전을 지정하고, 두 번째 셀은 `neuralforecast`, `datasetsforecast`를 설치합니다.

- `torch`: 텐서 연산과 딥러닝 학습의 기반입니다.
- `torchvision`, `torchaudio`: 각각 영상·오디오 도구이며 아래 시계열 코드에서 직접 사용하지 않습니다. 원본 설치 명령에 포함되어 있어 유지했습니다.
- `neuralforecast`: 모델 구현, 학습, 예측을 제공합니다.
- `datasetsforecast`: 시계열 벤치마크의 다운로드·로딩을 제공합니다.

`%%capture`는 출력과 경고를 숨기는 Jupyter cell magic입니다. 설치 실패나 import 문제가 보이지 않으면 해당 줄을 임시로 제거해 원인을 확인하세요. `!pip`는 셸 명령이며 일반 `.py` 문법과 다릅니다.

**버전 주의:** PyTorch는 고정되어 있지만 뒤의 두 패키지는 고정되어 있지 않습니다. 새 환경에서는 의존성이나 API가 달라질 수 있습니다. 기존 런타임의 PyTorch를 바꾼 뒤 재시작이 필요할 수도 있으므로 수업 전에 동일한 환경에서 설치·import를 점검하고 실제 버전을 기록하세요. 이번 편집은 설치 명령의 호환성까지 보장하는 수정이 아닙니다.

In [ ]:
%%capture
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

In [ ]:
%%capture
!pip install neuralforecast datasetsforecast

## 2. ETTm2 데이터 불러오기

ETTm2는 전력 변압기의 부하와 오일 온도 등을 15분 간격으로 기록한 시계열입니다. 여기서 데이터 설명의 **전력 변압기(transformer)**와 신경망 **Transformer**는 서로 다른 의미입니다.

`LongHorizon.load`는 `Y_df`, 외생변수용 DataFrame, 정적변수용 DataFrame을 반환합니다. 아래 코드의 `Y_df, _, _`는 첫 번째만 사용하고 나머지 두 반환값을 사용하지 않겠다는 뜻입니다.

이 wrapper의 공개 구현은 학습 구간 통계로 정규화된 벤치마크를 로드합니다. 따라서 `y`를 반드시 원시 센서 단위의 값으로 해석하면 안 됩니다. 원래 단위로 평가하려면 전처리 정보가 별도로 필요합니다. [LongHorizon 데이터 로더 소스](https://github.com/Nixtla/datasetsforecast/blob/main/datasetsforecast/long_horizon.py)

In [ ]:
import pandas as pd

from datasetsforecast.long_horizon import LongHorizon

### 2-1. Long format: 변수가 열이 아니라 시계열 ID가 된다

| 열 | 역할 | 예 |
|---|---|---|
| `unique_id` | 어떤 시계열인지 식별 | `OT`, `HUFL` 등 |
| `ds` | 관측 시점 | 15분 간격 timestamp |
| `y` | 그 시계열의 관측값 | 예측할 실수 값 |

일반적인 wide format에서는 날짜 한 행에 여러 센서 열이 있지만, 여기서는 각 센서가 별도 `unique_id`로 세로로 쌓입니다. “OT 예측 시 다른 센서 6개 값을 같은 입력 벡터로 쓴다”는 코드가 아닙니다.

`pd.to_datetime`은 `ds`를 날짜·시간 자료형으로 변환합니다. `Y_df.groupby('unique_id').head(2)`는 **시계열마다 앞의 두 행**을 출력합니다. 단순히 전체에서 두 행만 출력하는 `Y_df.head(2)`와 다릅니다.

자신의 데이터로 바꾸면 ID별 시간 정렬, 동일 간격, 누락·중복, 시계열별 길이를 확인하세요. 아래 `n_time` 계산은 모든 시계열이 공통 시간축을 갖는 데이터에 적합합니다.

In [ ]:
# Change this to your own data to try the model
Y_df, _, _ = LongHorizon.load(directory='./', group='ETTm2')
Y_df['ds'] = pd.to_datetime(Y_df['ds'])

n_time = len(Y_df.ds.unique())
val_size = int(.2 * n_time)
test_size = int(.2 * n_time)

Y_df.groupby('unique_id').head(2)

### 2-2. 학습·검증·테스트 길이 계산

`n_time = len(Y_df.ds.unique())`는 전체 행 수가 아니라 **고유 시점 수**입니다. 7개 시계열이 동일한 시간축을 갖는다면 전체 행 수는 그 7배입니다.

시간순으로 앞의 약 60%는 학습, 그다음 약 20%는 validation, 마지막 약 20%는 test에 해당하도록 길이를 지정합니다. `int()` 때문에 실제 개수에는 반올림이 아닌 내림이 적용됩니다. 아래 모델 학습 호출이 이 길이를 받아 구간을 처리합니다.

| 구간 | 사용하는 목적 |
|---|---|
| Train | 모델 가중치 학습 |
| Validation | 학습 중 손실 확인과 조기 종료 |
| Test | 학습·선택이 끝난 모델의 예측 평가 |

`val_size`와 `test_size`는 윈도우 개수나 전체 DataFrame의 행 개수가 아니라 **시계열별 끝부분 구간 길이**입니다. 외생변수 반환값을 버렸으므로 달력 특징 등을 모델 입력에 명시적으로 전달하지 않습니다.

## 3. 모델 생성과 학습 설정

`Informer`, `Autoformer`, `PatchTST`를 동일한 입력 길이·예측 길이·최대 업데이트 횟수로 생성합니다. **동일한 설정 일부를 사용한다고 모델 크기나 계산량까지 같아지는 것은 아닙니다.** 나머지 구조·최적화 설정은 각 클래스의 기본값에 의존합니다.

`NeuralForecast`는 DataFrame과 여러 모델을 함께 다루는 상위 인터페이스입니다. 직접 `loss.backward()`를 쓰지 않아도 학습 코드가 라이브러리 내부에서 실행됩니다.

`FEDformer`는 import 문에 있지만 `models` 목록에는 없습니다. 따라서 이번 실행에서는 FEDformer를 학습하거나 평가하지 않습니다. 제외 이유는 원본 실습에서의 학습 시간이며, 모든 환경에서의 절대적인 속도 순위라는 뜻은 아닙니다.

In [ ]:
%%capture
from neuralforecast.core import NeuralForecast
from neuralforecast.models import Informer, Autoformer, FEDformer, PatchTST

### 세 모델이 긴 시계열을 다루는 방식

| 모델 | 핵심 아이디어 | 읽을 때의 포인트 |
|---|---|---|
| Informer | ProbSparse Attention과 distilling | 모든 query를 같은 비용으로 처리하지 않고 긴 입력을 효율적으로 다루려는 설계 |
| Autoformer | 추세·계절 성분 분해와 Auto-Correlation | 서로 다른 시간 지연에서 반복되는 패턴을 활용 |
| PatchTST | 연속된 관측 구간을 패치 토큰으로 구성 | 시점 수보다 토큰 수를 줄이고 구간 패턴을 표현 |

기본 full attention은 입력 토큰 수 $L$에 대해 $L\times L$의 관계를 다룹니다. 패치로 토큰 수를 줄이거나 정보 결합 방식을 바꾸는 것은 긴 입력의 계산량을 줄이는 동기가 됩니다. 실제 속도는 구현, 하드웨어, 모델 크기, 패치 설정에 따라 달라집니다.

`11_1`에서는 길이 4를 두 구간으로 묶었지만, PatchTST의 패치 길이·stride·정규화 등은 별도 설계입니다. 두 구현이 같다고 생각하면 안 됩니다. 이번 코드는 Attention 가중치를 직접 반환하거나 시각화하지 않습니다.

공식 구조 설명: [Informer](https://nixtlaverse.nixtla.io/neuralforecast/models.informer.html), [Autoformer](https://nixtlaverse.nixtla.io/neuralforecast/models.autoformer.html), [PatchTST](https://nixtlaverse.nixtla.io/neuralforecast/models.patchtst.html).

### 3-1. `horizon=96`과 하이퍼파라미터 해석

ETTm2는 15분 간격이므로 다음과 같습니다.

$$96\times15\ \text{minutes}=1440\ \text{minutes}=24\ \text{hours}.$$

즉 **과거 24시간을 보고 앞으로 24시간을 예측**합니다. 원본 주석의 `4 * 15 min`은 1시간에 해당하고, 하루에는 그 24배인 96개 관측이 필요합니다.

| 인자 | 값 | 의미 |
|---|---:|---|
| `h` | 96 | 한 번에 예측할 미래 관측값 개수 |
| `input_size` | 96 | 각 예측에 사용할 과거 관측값 개수 |
| `max_steps` | 1000 | 최대 gradient 업데이트 횟수; 1000 epoch라는 뜻이 아님 |
| `val_check_steps` | 100 | 100번의 학습 step 간격으로 validation 확인 |
| `early_stop_patience_steps` | 3 | validation 개선이 없는 검사 횟수에 대한 patience |

patience의 3은 단순한 optimizer step 3번을 뜻하지 않습니다. 검사 간격이 100이면 대략 300 step 동안 개선이 없을 때 종료될 수 있으나, 정확한 검사·종료 시점은 학습 프레임워크 설정에 따릅니다. [모델 설정 API](https://nixtlaverse.nixtla.io/neuralforecast/models.informer.html)

모델 생성자는 네트워크와 학습 설정을 준비합니다. 아래 목록은 앙상블 평균을 내는 코드가 아니라 **세 모델의 예측을 각각 보관해 비교**하기 위한 목록입니다.

In [ ]:
%%capture
horizon = 96 # 24hrs = 4 * 15 min.
models = [Informer(h=horizon,                 # Forecasting horizon
                input_size=horizon,           # Input size
                max_steps=1000,               # Number of training iterations
                val_check_steps=100,          # Compute validation loss every 100 steps
                early_stop_patience_steps=3), # Stop training if validation loss does not improve
          Autoformer(h=horizon,
                input_size=horizon,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=3),
          PatchTST(h=horizon,
                input_size=horizon,
                max_steps=1000,
                val_check_steps=100,
                early_stop_patience_steps=3),
         ]

### Early stopping과 하이퍼파라미터 탐색은 다릅니다

현재 코드는 정해진 세 모델 설정에 대해 validation loss를 모니터링하고 조기 종료에 사용합니다. **`cross_validation`을 호출한다고 입력 길이·head 수·학습률을 자동으로 탐색하지는 않습니다.**

자동 탐색을 하려면 별도의 탐색 공간과 `Auto*` 모델 등의 튜닝 절차를 구성해야 합니다. 여러 설정을 비교할 때에는 validation으로 선택하고, 최종 test 결과를 보고 설정을 반복 선택하지 않도록 주의하세요. 원본의 특수 callout 표기는 일반 Jupyter에서 읽기 쉬운 한국어 설명으로 바꾸었습니다.

### 3-2. `NeuralForecast`와 시간순 교차검증

`NeuralForecast(models=models, freq='15min')`에서 `freq`는 관측 간격입니다. 예측 길이가 아니며 누락 구간을 자동으로 채우는 명령도 아닙니다.

아래 `cross_validation` 호출이 **실제 학습과 테스트 구간 예측을 수행하는 단계**입니다. 여기서 말하는 교차검증은 무작위 K-fold가 아니라 과거의 여러 예측 기준시점에서 미래를 예측하는 시계열 평가입니다.

| 인자 | 이 코드에서의 역할 |
|---|---|
| `df=Y_df` | 전체 시계열 전달 |
| `val_size=val_size` | 학습 구간 뒤의 validation 길이 지정 |
| `test_size=test_size` | 마지막 테스트 구간의 길이 지정 |
| `n_windows=None` | 테스트 길이와 horizon·간격으로 윈도우 개수를 정하도록 설정 |

현재 공개 API의 기본값은 `step_size=1`, `refit=False`입니다. 즉, 예측 기준시점을 한 관측 간격씩 옮기되 매번 모델을 재학습하는 설정은 아닙니다. 설치 버전의 기본값이 달라지면 동작도 달라질 수 있으므로 실제 실행에서는 확인이 필요합니다. [NeuralForecast 핵심 API](https://nixtlaverse.nixtla.io/neuralforecast/core.html)

### Rolling-origin 예측은 무엇을 반복하는가?

기준시점 $c$까지 관측했다고 가정하면, 한 예측의 입력은 $y_{c-95:c}$이고 출력은 $\hat y_{c+1:c+96}$입니다. 기준시점을 한 칸 옮기면 입력에 새로 관측된 실제값을 포함하여 다음 96개를 다시 예측합니다.

| 기준시점 | 이용 가능한 과거 | 평가할 미래 |
|---|---|---|
| $c$ | $c-95$부터 $c$까지 | $c+1$부터 $c+96$까지 |
| $c+1$ | $c-94$부터 $c+1$까지 | $c+2$부터 $c+97$까지 |
| $c+2$ | $c-93$부터 $c+2$까지 | $c+3$부터 $c+98$까지 |

뒤의 cutoff에서는 그때까지 실제로 관측되었다고 보는 테스트 구간의 과거값을 입력으로 쓸 수 있습니다. 단, **현재 cutoff 이후의 실제값은 그 예측의 입력이 아니라 평가용 정답**입니다. 따라서 테스트 전체를 첫날 한 번에 예측하는 실험도, 각 윈도우마다 반드시 재학습하는 실험도 아닙니다.

공통 시간축과 누락 없는 데이터, `step_size=1`, 전체 horizon이 들어가는 윈도우라는 조건에서 cutoff 수는

$$W=\text{test\_size}-H+1$$

입니다. 예를 들어 테스트 길이가 11,520이면 $W=11,425$이고, 시계열 7개·horizon 96에서는 결과 행이 $7\times11,425\times96=7,677,600$개입니다. **이는 조건에 따른 계산 예시이지 이번 실행으로 확인한 행 수가 아닙니다.** 학습 데이터보다 결과 테이블이 훨씬 커질 수 있는 이유입니다.

In [ ]:
nf = NeuralForecast(
    models=models,
    freq='15min')

Y_hat_df = nf.cross_validation(df=Y_df,
                               val_size=val_size,
                               test_size=test_size,
                               n_windows=None)

### 3-3. 예측 결과 DataFrame 읽기

`Y_hat_df.head()`는 앞의 몇 행을 확인합니다. 전체 결과나 평균 성능을 보여주는 코드는 아닙니다.

| 열 | 의미 |
|---|---|
| `unique_id` | 예측 대상 시계열 이름 |
| `ds` | 정답·예측이 해당하는 미래 시점 |
| `cutoff` | 그 예측이 출발한 기준시점 |
| `y` | 해당 미래 시점의 실제값 |
| `Informer`, `Autoformer`, `PatchTST` | 각 모델의 예측값 |

**한 행은 “특정 시계열·특정 cutoff에서 특정 미래 시점을 예측한 결과”입니다.** 같은 `unique_id`와 `ds`라도 cutoff가 다르면 다른 예측입니다. 그러므로 중복 날짜가 있다고 곧바로 중복 행을 제거하면 평가의 의미가 달라집니다.

버전에 따라 `unique_id`가 열이 아닌 인덱스로 반환되는 경우에는 뒤의 열 기반 필터링 전에 결과 구조를 확인하고 `reset_index()` 등을 검토해야 합니다. 이 파일에서는 원본 코드를 바꾸지 않았습니다.

In [ ]:
Y_hat_df.head()

## 4. 결과 시각화와 MAE 평가

그래프는 특정 시계열의 패턴을 살펴보는 정성적 평가이고, MAE는 예측 오차를 수치로 요약하는 정량적 평가입니다. **아래 그래프는 OT만, 아래 MAE는 `Y_hat_df`의 모든 시계열·모든 예측 윈도우를 사용**한다는 차이가 중요합니다.

### 4-1. OT 변수의 예측 그래프

`OT`는 오일 온도 시계열의 ID입니다. 아래 코드는 먼저 OT 행을 고른 뒤, 서로 가까운 예측 윈도우가 겹쳐 보이는 것을 줄이기 위해 cutoff를 일부 선택합니다.

`cutoffs = ...unique()[::horizon]`는 **고유 cutoff 목록에서 96개마다 하나를 고르는 슬라이싱**입니다. 매 cutoff의 96번째 예측만 고르는 것이 아닙니다. cutoff가 시간순이고 15분 간격이면 하루 간격의 기준시점을 선택하는 셈입니다.

이 선택은 이미 학습·예측한 결과를 **그림용으로 추리는 것**이며, 앞의 학습이나 전체 MAE에 사용하는 `Y_hat_df`를 바꾸지는 않습니다.

In [ ]:
import matplotlib.pyplot as plt

#### 그래프 코드의 인덱싱 주의사항

원본의 다음 줄은 이미 OT만 남긴 `Y_plot`을 다시 필터링하면서, 마스크는 전체 `Y_hat_df`에서 만들고 있습니다.

`Y_plot = Y_plot[Y_hat_df['cutoff'].isin(cutoffs)]`

원래 인덱스가 고유하고 보존되어 있으면 pandas가 인덱스를 맞추어 처리할 수 있지만, 재정렬 경고가 발생하거나 인덱스 조건에 따라 문제가 생길 수 있습니다. 수정할 때는 **`Y_plot`에서 만든 마스크로 `Y_plot`을 필터링**하는 편이 명확합니다. 또한 cutoff 목록을 시간순 정렬한 뒤 일정 간격으로 고르고, 결과를 `cutoff`, `ds` 순으로 정렬하는 것이 안전합니다.

`start=0`, `end=500`은 표시할 Series의 앞부분 슬라이스입니다. 절대 시각 0부터 500까지나 모델의 예측 horizon 500을 뜻하지 않습니다. 행 기준으로 자르려는 의도는 `iloc[start:end]`를 쓰면 더 명확합니다. 원본 코드는 유지했습니다.

In [ ]:
Y_plot = Y_hat_df[Y_hat_df['unique_id']=='OT'] # OT dataset
cutoffs = Y_hat_df['cutoff'].unique()[::horizon]
Y_plot = Y_plot[Y_hat_df['cutoff'].isin(cutoffs)]

start = 0
end = 500

plt.figure(figsize=(20,5))
plt.plot(Y_plot['ds'][start:end], Y_plot['y'][start:end], label='True')
plt.plot(Y_plot['ds'][start:end], Y_plot['Informer'][start:end], label='Informer')
plt.plot(Y_plot['ds'][start:end], Y_plot['Autoformer'][start:end], label='Autoformer')
plt.plot(Y_plot['ds'][start:end], Y_plot['PatchTST'][start:end], label='PatchTST')
plt.xlabel('Datestamp')
plt.ylabel('OT')
plt.grid()
plt.legend()

#### 그래프에서 확인할 점

- `True`와 각 예측이 상승·하락 방향을 함께 따라가는지 봅니다.
- 피크가 평평하게 예측되거나 시점이 늦어지는지 확인합니다.
- 여러 cutoff의 예측을 이어 그리므로, 윈도우 경계에서 선이 꺾이는 것이 센서의 급변인지 예측 시작점이 바뀐 효과인지 구분합니다.
- 이 그림은 OT 일부 행만 보여줍니다. 전체 시계열에서 어떤 모델이 항상 좋다는 결론은 낼 수 없습니다.

별도로 Attention 행렬을 그린 것이 아니므로, 예측선만 보고 어느 과거 시점에 Attention을 집중했는지 판단할 수는 없습니다.

### 4-2. Mean Absolute Error: 평균 절대 오차

아래 코드는 예측과 실제값의 절대 차이를 모든 결과 행에 대해 평균냅니다.

$$\mathrm{MAE}=\frac{1}{N_{\mathrm{rows}}}
\sum_{r=1}^{N_{\mathrm{rows}}}|y_r-\hat y_r|.$$

균형 잡힌 패널에서 시계열 수 $S$, cutoff 수 $W$, horizon $H$라면 같은 계산을 다음처럼 쓸 수 있습니다.

$$\mathrm{MAE}=\frac{1}{SWH}\sum_{s=1}^{S}\sum_{w=1}^{W}\sum_{k=1}^{H}
\left|y_{s,c_w+k}-\hat y^{(w)}_{s,c_w+k}\right|.$$

작을수록 평균 절대 오차가 작습니다. 예를 들어 실제값 1.0, 예측값 0.7인 행의 기여는 0.3입니다. 부호를 없애므로 과대·과소 예측이 서로 상쇄되지 않습니다. 이는 정확도 퍼센트가 아닙니다.

현재 코드의 MAE는 **OT만의 오차가 아니며**, 정규화된 `Y_df` 척도에서 계산됩니다. 윈도우가 겹치면 같은 실제 시점도 서로 다른 cutoff·예측 거리의 사례로 여러 번 포함됩니다. 단순히 “고유한 테스트 날짜당 한 번씩 계산한 평균”과는 다릅니다.

#### 평가 함수 import의 버전 의존성

아래 `from neuralforecast.losses.numpy import mae`는 원본이 사용한 경로입니다. 설치되는 버전에서 해당 경로가 없다면 이 셀에서 import 오류가 날 수 있습니다. 현재 공식 튜토리얼은 `utilsforecast` 기반 평가 경로를 사용하므로 **이전 import와 최신 설치 명령이 함께 있는 점을 수업 전에 확인**해야 합니다. [원본 튜토리얼의 평가 절](https://nixtlaverse.nixtla.io/neuralforecast/docs/tutorials/longhorizon_transformers.html)

이는 수식이 바뀐다는 뜻은 아닙니다. API를 바꿀 때에도 현재처럼 전체 행을 평균낼지, 시계열별 MAE를 먼저 계산한 뒤 평균낼지 명시해야 합니다. 시계열별 행 수가 다르면 두 집계 결과는 달라질 수 있습니다.

In [ ]:
from neuralforecast.losses.numpy import mae

In [ ]:
mae_informer = mae(Y_hat_df['y'], Y_hat_df['Informer'])
mae_autoformer = mae(Y_hat_df['y'], Y_hat_df['Autoformer'])
mae_patchtst = mae(Y_hat_df['y'], Y_hat_df['PatchTST'])

print(f'Informer: {mae_informer:.3f}')
print(f'Autoformer: {mae_autoformer:.3f}')
print(f'PatchTST: {mae_patchtst:.3f}')

#### 출력되는 숫자 읽기

세 줄은 동일한 `Y_hat_df['y']`에 대한 각 모델의 MAE를 소수점 셋째 자리까지 표시합니다. 낮을수록 이 평가 설정에서 평균 절대 오차가 작다는 의미입니다. 소수점 반올림으로 작은 차이가 가려질 수 있습니다.

전체 행 평균에서는 **예측 행이 더 많은 시계열이 더 큰 가중치**를 갖습니다. 모든 시계열의 행 수가 같으면 시계열별 MAE를 동일 비중으로 평균낸 값과 일치합니다.

이 셀은 학습 시간, 파라미터 수, 예측 구간별 성능, 불확실성 구간을 비교하지 않습니다. MAE 하나로 계산 효율이나 신뢰성까지 평가했다고 볼 수 없습니다.

### 4-3. 원본에 실린 논문 참고표 — 이번 실행 결과 아님

아래 숫자는 **원본 노트북에서 가져온 문헌 비교용 표를 그대로 유지한 것**입니다. 이번 수정 과정에서 다시 산출하거나 각 논문의 평가 조건을 재현하여 검증한 수치가 아닙니다. 직전 셀에서 출력하는 세 모델의 MAE와 혼동하지 마세요.

| 예측 길이 H | PatchTST | Autoformer | Informer | ARIMA |
|---|---:|---:|---:|---:|
| 96 | 0.256 | 0.339 | 0.453 | 0.301 |
| 192 | 0.296 | 0.340 | 0.563 | 0.345 |
| 336 | 0.329 | 0.372 | 0.887 | 0.386 |
| 720 | 0.385 | 0.419 | 1.388 | 0.445 |

현재 코드는 `horizon=96` 한 가지 설정만 실행하며, ARIMA도 학습하지 않습니다. 나머지 horizon 행은 이 코드의 실행 결과가 아닙니다. 이 표를 근거로 직접적인 성능 우위를 주장하려면 같은 데이터 버전, 시간 분할, 입력 길이, 정규화, 변수 범위, 평가 척도·집계 방식인지 먼저 확인해야 합니다.

## 5. 확장 실습과 정리

다음은 기존 코드를 변경하여 해볼 확장 과제입니다. 이 파일에는 자동으로 적용되어 있지 않습니다.

1. **입력 길이와 예측 길이 분리:** `h=96`은 유지하고 `input_size`만 늘려 더 긴 과거를 보는 효과를 비교합니다. 두 값을 동시에 바꾸면 효과를 구분하기 어렵습니다.
2. **예측 거리별 오차:** cutoff에서 1 step 뒤와 96 step 뒤의 MAE를 각각 비교합니다. 전체 평균은 먼 미래에서 커지는 오차를 숨길 수 있습니다.
3. **시계열별 평가:** `unique_id`별 MAE를 계산하여 OT의 그래프와 OT의 수치가 같은 대상을 설명하도록 합니다.
4. **기준선 추가:** 직전 값을 반복하거나 전날 패턴을 사용하는 단순 기준선과 비교합니다. Attention을 사용한다는 이유만으로 우월하다고 가정하지 않습니다.
5. **다른 모델 비교:** 원본에서 소개한 feed-forward 계열 NHITS를 비교 대상으로 추가할 수 있습니다. 이번 노트북에서는 학습하지 않았으며 정확도·속도 우위도 실험으로 확인해야 합니다. [NHITS 예제](https://nixtlaverse.nixtla.io/neuralforecast/docs/tutorials/longhorizon_nhits.html)

비교할 때 데이터 분할을 고정하고 validation을 이용해 설정을 선택하세요. 입력 길이·모델 복잡도가 커지면 수업 실행 시간이 늘어날 수 있습니다.

**한 줄 정리:** 과거 96개 관측으로 다음 96개를 예측하는 세 모델을, 여러 cutoff에서 비교하는 라이브러리 활용 실습입니다.

### 두 실습의 차이 요약

| 항목 | 11-1 직접 구현 | 11-2 라이브러리 활용 |
|---|---|---|
| 입력 → 출력 | 과거 4개월 → 다음 1개월 | 과거 96개 15분 관측 → 다음 96개 |
| 모델 | 작은 Encoder 기반 회귀 모델 | Informer·Autoformer·PatchTST |
| 입력 구성 | 4개 값을 2개 토큰으로 직접 reshape | 모델별 내부 임베딩·패치 처리 |
| 학습 | 직접 MSE·Adam·역전파 작성 | 라이브러리의 모델 학습 절차 |
| 평가 | 실제 과거를 쓰는 rolling 1-step | cutoff를 옮기는 multi-horizon 평가 |
| 주의점 | test 기반 선택·최적 가중치 참조 문제 | API 버전·겹치는 예측·지표 집계 범위 |

다음 질문에 답하면 핵심을 이해한 것입니다. **“입력 길이와 예측 길이는 각각 얼마인가?”, “예측 시점에 실제로 알 수 있는 값만 입력에 들어가는가?”, “출력 MAE는 어떤 행들을 평균한 값인가?”**

## 참고문헌

[Zhou, H., Zhang, S., Peng, J., Zhang, S., Li, J., Xiong, H., & Zhang, W. (2021, May). Informer: Beyond efficient transformer for long sequence time-series forecasting. In Proceedings of the AAAI conference on artificial intelligence (Vol. 35, No. 12, pp. 11106-11115)](https://ojs.aaai.org/index.php/AAAI/article/view/17325)

[Wu, H., Xu, J., Wang, J., & Long, M. (2021). Autoformer: Decomposition transformers with auto-correlation for long-term series forecasting. Advances in Neural Information Processing Systems, 34, 22419-22430.](https://proceedings.neurips.cc/paper/2021/hash/bcc0d400288793e8bdcd7c19a8ac0c2b-Abstract.html)

[Zhou, T., Ma, Z., Wen, Q., Wang, X., Sun, L., & Jin, R. (2022, June). Fedformer: Frequency enhanced decomposed transformer for long-term series forecasting. In International Conference on Machine Learning (pp. 27268-27286). PMLR.](https://proceedings.mlr.press/v162/zhou22g.html)


[Nie, Y., Nguyen, N. H., Sinthong, P., & Kalagnanam, J. (2022). A Time Series is Worth 64 Words: Long-term Forecasting with Transformers.](https://arxiv.org/pdf/2211.14730.pdf)

[Cristian Challu, Kin G. Olivares, Boris N. Oreshkin, Federico Garza, Max Mergenthaler-Canseco, Artur Dubrawski (2021). NHITS: Neural Hierarchical Interpolation for Time Series Forecasting. Accepted at AAAI 2023.](https://arxiv.org/abs/2201.12886)

### 설명 보강 시 확인한 자료

- 첨부 강의자료: **D06 Transformer2026R.pdf**, Q/K/V·Multi-Head Attention·Encoder block·위치 정보·계산 복잡도 관련 부분.
- 코드 동작·현재 API 참고: [NeuralForecast Core](https://nixtlaverse.nixtla.io/neuralforecast/core.html), [LongHorizon 로더](https://github.com/Nixtla/datasetsforecast/blob/main/datasetsforecast/long_horizon.py).
- 원본 논문 목록은 위에 유지했습니다. 문헌의 모델 설명과 이 라이브러리의 실제 구현·기본값은 구분해서 확인하세요.

웹 문서는 설명 확인용입니다. 설치 패키지가 고정되지 않았으므로, 수업에서 사용한 라이브러리 버전과 실행 설정을 함께 기록해야 결과를 해석하기 쉽습니다.